# Inference on Test Datasets
This notebook performs inference on the MultiNLI dev_matched and dev_mismatched datasets.

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import joblib
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

## Configuration

In [ ]:
# Specify the experiment number to use for inference
EXP_NUMBER = 27  # Change this to the experiment you want to test

artifacts_path = f"../exp_results/artifacts/{EXP_NUMBER}"
inference_results_path = f"../exp_results/artifacts/{EXP_NUMBER}/inference_results"

# Create inference results folder
os.makedirs(inference_results_path, exist_ok=True)

print(f"Using experiment number: {EXP_NUMBER}")
print(f"Artifacts path: {artifacts_path}")
print(f"Results will be saved to: {inference_results_path}")

## Load Test Datasets

In [ ]:
# Load dev_matched and dev_mismatched datasets
dev_matched_path = "../data/raw/multinli_1.0_dev_matched.jsonl"
dev_mismatched_path = "../data/raw/multinli_1.0_dev_mismatched.jsonl"

# Read JSONL files
dev_matched = pd.read_json(dev_matched_path, lines=True)
dev_mismatched = pd.read_json(dev_mismatched_path, lines=True)

print(f"Dev matched dataset shape: {dev_matched.shape}")
print(f"Dev mismatched dataset shape: {dev_mismatched.shape}")

# Display first few rows
print("\nDev matched sample:")
display(dev_matched.head())

print("\nDev mismatched sample:")
display(dev_mismatched.head())

In [ ]:
# Check for gold_label column and filter if needed
print("Dev matched gold_label distribution:")
print(dev_matched['gold_label'].value_counts())

print("\nDev mismatched gold_label distribution:")
print(dev_mismatched['gold_label'].value_counts())

# Remove entries with '-' label if present
dev_matched = dev_matched[dev_matched['gold_label'] != '-'].copy()
dev_mismatched = dev_mismatched[dev_mismatched['gold_label'] != '-'].copy()

print(f"\nAfter filtering - Dev matched: {dev_matched.shape}")
print(f"After filtering - Dev mismatched: {dev_mismatched.shape}")

## Text Preprocessing Functions

In [ ]:
import spacy
import re
import unicodedata
from tqdm import tqdm

nlp = spacy.load(
    "en_core_web_sm",
    disable=["parser", "ner"]
)
nlp.max_length = 10_000_000

def clean_text(text):
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    text = re.sub(r"[^a-zA-Z\s]", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

def preprocess_texts(texts, batch_size=500, n_process=-1):
    texts = [clean_text(t) for t in texts]
    
    cleaned = []
    for doc in tqdm(
        nlp.pipe(texts, batch_size=batch_size, n_process=n_process),
        total=len(texts),
        desc="Preprocessing"
    ):
        tokens = [token.lemma_ for token in doc]
        cleaned.append(" ".join(tokens))

    return cleaned

## Preprocess Test Data

In [ ]:
# Preprocess dev_matched
print("Preprocessing dev_matched sentences...")
dev_matched['sentence1_cleaned'] = preprocess_texts(dev_matched['sentence1'].astype(str).tolist())
dev_matched['sentence2_cleaned'] = preprocess_texts(dev_matched['sentence2'].astype(str).tolist())

print("\nPreprocessing dev_mismatched sentences...")
dev_mismatched['sentence1_cleaned'] = preprocess_texts(dev_mismatched['sentence1'].astype(str).tolist())
dev_mismatched['sentence2_cleaned'] = preprocess_texts(dev_mismatched['sentence2'].astype(str).tolist())

print("\nPreprocessing completed!")

## Load Pre-trained TF-IDF Vectorizer

In [ ]:
# Load the TF-IDF vectorizer from training
vectorizer_path = f"{artifacts_path}/vectorizer/tfidf_vectorizer.pkl"

if os.path.exists(vectorizer_path):
    tfidf = joblib.load(vectorizer_path)
    print(f"Loaded TF-IDF vectorizer from {vectorizer_path}")
    print(f"Vocabulary size: {len(tfidf.vocabulary_)}")
else:
    print(f"ERROR: Vectorizer not found at {vectorizer_path}")
    print("Please ensure the model and vectorizer were saved during training.")

## Apply TF-IDF Transformation

In [ ]:
def apply_tfidf_transform(df, tfidf):
    """Apply TF-IDF transformation to sentence1 and sentence2"""
    tfidf_s1 = tfidf.transform(df["sentence1_cleaned"]).toarray()
    tfidf_s2 = tfidf.transform(df["sentence2_cleaned"]).toarray()
    
    # Create feature names
    feature_names_s1 = [f"s1_{w}" for w in tfidf.get_feature_names_out()]
    feature_names_s2 = [f"s2_{w}" for w in tfidf.get_feature_names_out()]
    tf_idf_features = feature_names_s1 + feature_names_s2
    
    # Create DataFrame with TF-IDF features
    features_df = pd.DataFrame(
        np.hstack([tfidf_s1, tfidf_s2]),
        columns=tf_idf_features
    )
    
    # Concatenate with original data
    result_df = pd.concat([df.reset_index(drop=True), features_df.reset_index(drop=True)], axis=1)
    
    return result_df, tf_idf_features

# Apply TF-IDF to both datasets
print("Applying TF-IDF transformation to dev_matched...")
dev_matched_tfidf, tf_idf_features = apply_tfidf_transform(dev_matched, tfidf)

print("Applying TF-IDF transformation to dev_mismatched...")
dev_mismatched_tfidf, _ = apply_tfidf_transform(dev_mismatched, tfidf)

print(f"\nTF-IDF features created: {len(tf_idf_features)}")
print(f"Dev matched shape after TF-IDF: {dev_matched_tfidf.shape}")
print(f"Dev mismatched shape after TF-IDF: {dev_mismatched_tfidf.shape}")

## Feature Engineering

In [ ]:
# Length features
def add_length_features(df):
    df['s1_length'] = df['sentence1_cleaned'].apply(lambda x: len(str(x).split()))
    df['s2_length'] = df['sentence2_cleaned'].apply(lambda x: len(str(x).split()))
    return df

dev_matched_tfidf = add_length_features(dev_matched_tfidf)
dev_mismatched_tfidf = add_length_features(dev_mismatched_tfidf)

print("Length features added.")

In [ ]:
# Negation features
negation_words = set(['not', 'no', 'nor', 'never', 'neither', 'none',
    'nobody', 'nothing', 'nowhere', 'noone',
    'without', 'hardly', 'scarcely', 'barely', 'seldom', 'rarely',
    'lack',
    'deny', 'refuse', 'reject', 'fail', 'avoid', 'prevent',
    'stop', 'exclude', 'oppose', 'disagree', 'neglect',
    'omit', 'miss', 'dismiss', 'discard', 'abandon',
    'cancel', 'block', 'ban', 'forbid', 'prohibit',
    'restrict', 'limit', 'decline', 'ignore',
    'contradict', 'negate', 'nullify', 'invalidate', 'revoke',
    'withdraw', 'withhold', 'suppress', 'conceal', 'hide',
    'impossible', 'unable', 'unlikely', 'insufficient', 'invalid',
    'incorrect', 'false', 'wrong', 'absent', 'missing',
    'incomplete', 'inaccurate', 'ineffective', 'inefficient',
    'unnecessary', 'unacceptable', 'unavailable', 'uncertain',
    'unclear', 'unaware', 'unfit', 'unsafe', 'unstable',
    'unsuccessful', 'unsupported', 'unwilling', 'unrelated',
    'unreliable', 'unreasonable', 'inappropriate', 'inadequate',
    'improper', 'impractical', 'improbable', 'imperfect',
    'powerless', 'helpless', 'useless', 'worthless', 'pointless',
    'hopeless', 'meaningless', 'baseless', 'groundless', 'senseless',
    'fake', 'void', 'null', 'defunct', 'obsolete',
    'failure', 'absence', 'loss', 'denial',
    'refusal', 'rejection', 'impossibility', 'prohibition',
    'restriction', 'limitation', 'contradiction', 'error',
    'mistake', 'fault', 'flaw', 'defect', 'gap',
    'can not', 'will not', 'do not', 'did not', 'be not',
    'have not', 'would not', 'should not', 'could not',
    'must not', 'might not', 'need not', 'shall not',
    'dare not', 'ought not', 'may not',
    'not only', 'not just', 'not even', 'not yet', 'not always',
    'not really', 'not quite', 'not enough', 'not sure', 'not true',
    'not work', 'not help', 'not support', 'not allow', 'not include',
    'not have', 'not be', 'not do', 'not go', 'not make',
    'not give', 'not take', 'not know', 'not want', 'not need',
    'not say', 'not mean', 'not show', 'not seem', 'not appear',
    'not exist', 'not apply', 'not relate', 'not match', 'not fit',
    'no long', 'no more', 'no way', 'no such', 'no need',
    'no effect', 'no result', 'no evidence', 'no proof',
    'no sign', 'no reason', 'no chance', 'no option', 'no solution',
    'no difference', 'no impact', 'no benefit', 'no use', 'no point',
    'no doubt', 'no question', 'no problem', 'no issue', 'no concern',
    'never again', 'never be', 'never have', 'never do',
    'never say', 'never show', 'never work', 'never happen',
    'never allow', 'never support',
    'neither nor',
    'by no', 'in no', 'on no',
    'far from', 'free from', 'rather than',
    'instead of', 'as oppose', 'let alone',
    'without any', 'without much', 'without enough',
    'without proper', 'without clear', 'without good',
    'fail to', 'lack of', 'deny any', 'refuse to',
    'unable to', 'hard to', 'difficult to', 'impossible to'])

def count_negation_words(text):
    words = str(text).lower().split()
    return sum(1 for word in words if word in negation_words)

def add_negation_features(df):
    df['s1_negation_count'] = df['sentence1_cleaned'].apply(count_negation_words)
    df['s2_negation_count'] = df['sentence2_cleaned'].apply(count_negation_words)
    return df

dev_matched_tfidf = add_negation_features(dev_matched_tfidf)
dev_mismatched_tfidf = add_negation_features(dev_mismatched_tfidf)

print("Negation features added.")

In [ ]:
# Shared words feature
def shared_word_percentage(row):
    s1_words = set(str(row['sentence1_cleaned']).split())
    s2_words = set(str(row['sentence2_cleaned']).split())
    if len(s1_words) == 0 or len(s2_words) == 0:
        return 0.0
    shared_words = s1_words.intersection(s2_words)
    total_words = s1_words.union(s2_words)
    return len(shared_words) / len(total_words) if len(total_words) > 0 else 0.0

print("Computing shared word ratios for dev_matched...")
dev_matched_tfidf['ratio_word_shared_sentence1_sentence2'] = dev_matched_tfidf.apply(shared_word_percentage, axis=1)

print("Computing shared word ratios for dev_mismatched...")
dev_mismatched_tfidf['ratio_word_shared_sentence1_sentence2'] = dev_mismatched_tfidf.apply(shared_word_percentage, axis=1)

print("Shared word features added.")

In [ ]:
# Antonym features
import nltk
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')
    nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()

def count_antonyms(row):
    s1_words = [lemmatizer.lemmatize(w.lower()) for w in str(row['sentence1_cleaned']).split()]
    s2_words = [lemmatizer.lemmatize(w.lower()) for w in str(row['sentence2_cleaned']).split()]
    
    total_words = len(set(s1_words) | set(s2_words)) if len(set(s1_words) | set(s2_words)) > 0 else 1
    
    antonym_count = 0
    
    for word in s1_words:
        antonyms = set()
        for syn in wordnet.synsets(word):
            for lemma in syn.lemmas():
                for ant in lemma.antonyms():
                    antonyms.add(lemmatizer.lemmatize(ant.name().lower()))
        
        if any(ant in s2_words for ant in antonyms):
            antonym_count += 1
            
    return antonym_count / total_words

print("Computing antonym ratios for dev_matched...")
dev_matched_tfidf['ratio_antonym_sentence1_sentence2'] = dev_matched_tfidf.apply(count_antonyms, axis=1)

print("Computing antonym ratios for dev_mismatched...")
dev_mismatched_tfidf['ratio_antonym_sentence1_sentence2'] = dev_mismatched_tfidf.apply(count_antonyms, axis=1)

print("Antonym features added.")

In [ ]:
# Combined ratio features
def add_ratio_features(df):
    df["ratio_length_sentence1_sentence2"] = df["s1_length"] / (df["s2_length"] + 1)
    df['ratio_negation_count_sentence1_sentence2'] = df['s1_negation_count'] / (df['s2_negation_count'] + 1)
    return df

dev_matched_tfidf = add_ratio_features(dev_matched_tfidf)
dev_mismatched_tfidf = add_ratio_features(dev_mismatched_tfidf)

print("Ratio features added.")

In [ ]:
# Cosine similarity feature (if using TF-IDF vectors)
from sklearn.metrics.pairwise import cosine_similarity

def add_cosine_similarity(df, tfidf):
    """Add cosine similarity between sentence1 and sentence2"""
    tfidf_s1 = tfidf.transform(df["sentence1_cleaned"])
    tfidf_s2 = tfidf.transform(df["sentence2_cleaned"])
    
    # Compute cosine similarity row by row
    cosine_sims = []
    for i in range(tfidf_s1.shape[0]):
        sim = cosine_similarity(tfidf_s1[i:i+1], tfidf_s2[i:i+1])[0][0]
        cosine_sims.append(sim)
    
    df['cosine_similarity_sentence1_sentence2'] = cosine_sims
    return df

print("Computing cosine similarity for dev_matched...")
dev_matched_tfidf = add_cosine_similarity(dev_matched_tfidf, tfidf)

print("Computing cosine similarity for dev_mismatched...")
dev_mismatched_tfidf = add_cosine_similarity(dev_mismatched_tfidf, tfidf)

print("Cosine similarity features added.")

## Prepare Features for Prediction

In [ ]:
# Select the same features used during training
added_features = [
    "ratio_word_shared_sentence1_sentence2",
    "ratio_length_sentence1_sentence2",
    "ratio_negation_count_sentence1_sentence2",
    "ratio_antonym_sentence1_sentence2",
    "s2_negation_count",
    "s2_length"
]

features_to_use = tf_idf_features + added_features

# Prepare features and labels
X_matched = dev_matched_tfidf[features_to_use]
y_matched = dev_matched_tfidf['gold_label']

X_mismatched = dev_mismatched_tfidf[features_to_use]
y_mismatched = dev_mismatched_tfidf['gold_label']

print(f"Dev matched features shape: {X_matched.shape}")
print(f"Dev mismatched features shape: {X_mismatched.shape}")
print(f"\nTotal features used: {len(features_to_use)}")

## Load Trained Model

In [ ]:
# Load the trained model
model_path = f"{artifacts_path}/model/model.pkl"

if os.path.exists(model_path):
    model = joblib.load(model_path)
    print(f"Loaded model from {model_path}")
    print(f"Model type: {type(model).__name__}")
else:
    print(f"ERROR: Model not found at {model_path}")
    print("Please ensure the model was saved during training.")

## Make Predictions

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Check if model is XGBoost (needs label encoding)
model_name = type(model).__name__

if 'XGB' in model_name:
    # XGBoost requires encoded labels
    label_encoder = LabelEncoder()
    label_encoder.fit(['contradiction', 'entailment', 'neutral'])  # Fit with expected classes
    
    print("Making predictions on dev_matched...")
    y_pred_matched_enc = model.predict(X_matched)
    y_pred_matched = label_encoder.inverse_transform(y_pred_matched_enc)
    
    print("Making predictions on dev_mismatched...")
    y_pred_mismatched_enc = model.predict(X_mismatched)
    y_pred_mismatched = label_encoder.inverse_transform(y_pred_mismatched_enc)
else:
    print("Making predictions on dev_matched...")
    y_pred_matched = model.predict(X_matched)
    
    print("Making predictions on dev_mismatched...")
    y_pred_mismatched = model.predict(X_mismatched)

print("\nPredictions completed!")

## Evaluate Results - Dev Matched

In [ ]:
# Evaluation metrics for dev_matched
print("=" * 60)
print("DEV MATCHED RESULTS")
print("=" * 60)

accuracy_matched = accuracy_score(y_matched, y_pred_matched)
precision_matched = precision_score(y_matched, y_pred_matched, average='weighted')
recall_matched = recall_score(y_matched, y_pred_matched, average='weighted')
f1_matched = f1_score(y_matched, y_pred_matched, average='weighted')

print(f"Accuracy: {accuracy_matched:.4f}")
print(f"Precision (weighted): {precision_matched:.4f}")
print(f"Recall (weighted): {recall_matched:.4f}")
print(f"F1-Score (weighted): {f1_matched:.4f}")

print("\nClassification Report:")
print(classification_report(y_matched, y_pred_matched))

In [ ]:
# Confusion matrix for dev_matched
cm_matched = confusion_matrix(y_matched, y_pred_matched)
class_names = sorted(y_matched.unique())

plt.figure(figsize=(10, 8))
sns.heatmap(cm_matched, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title(f'Confusion Matrix - Dev Matched\nAccuracy: {accuracy_matched:.4f}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{inference_results_path}/confusion_matrix_dev_matched.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"Confusion matrix saved to {inference_results_path}/confusion_matrix_dev_matched.png")

## Evaluate Results - Dev Mismatched

In [ ]:
# Evaluation metrics for dev_mismatched
print("=" * 60)
print("DEV MISMATCHED RESULTS")
print("=" * 60)

accuracy_mismatched = accuracy_score(y_mismatched, y_pred_mismatched)
precision_mismatched = precision_score(y_mismatched, y_pred_mismatched, average='weighted')
recall_mismatched = recall_score(y_mismatched, y_pred_mismatched, average='weighted')
f1_mismatched = f1_score(y_mismatched, y_pred_mismatched, average='weighted')

print(f"Accuracy: {accuracy_mismatched:.4f}")
print(f"Precision (weighted): {precision_mismatched:.4f}")
print(f"Recall (weighted): {recall_mismatched:.4f}")
print(f"F1-Score (weighted): {f1_mismatched:.4f}")

print("\nClassification Report:")
print(classification_report(y_mismatched, y_pred_mismatched))

In [ ]:
# Confusion matrix for dev_mismatched
cm_mismatched = confusion_matrix(y_mismatched, y_pred_mismatched)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_mismatched, annot=True, fmt='d', cmap='Greens', 
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title(f'Confusion Matrix - Dev Mismatched\nAccuracy: {accuracy_mismatched:.4f}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{inference_results_path}/confusion_matrix_dev_mismatched.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"Confusion matrix saved to {inference_results_path}/confusion_matrix_dev_mismatched.png")

## Compare Results

In [ ]:
# Comparison table
results_comparison = pd.DataFrame({
    'Dataset': ['Dev Matched', 'Dev Mismatched'],
    'Accuracy': [accuracy_matched, accuracy_mismatched],
    'Precision': [precision_matched, precision_mismatched],
    'Recall': [recall_matched, recall_mismatched],
    'F1-Score': [f1_matched, f1_mismatched]
})

print("\n" + "=" * 60)
print("COMPARISON OF RESULTS")
print("=" * 60)
display(results_comparison)

# Visualize comparison
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(results_comparison))
width = 0.2

ax.bar(x - 1.5*width, results_comparison['Accuracy'], width, label='Accuracy', alpha=0.8)
ax.bar(x - 0.5*width, results_comparison['Precision'], width, label='Precision', alpha=0.8)
ax.bar(x + 0.5*width, results_comparison['Recall'], width, label='Recall', alpha=0.8)
ax.bar(x + 1.5*width, results_comparison['F1-Score'], width, label='F1-Score', alpha=0.8)

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(results_comparison['Dataset'])
ax.legend()
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f"{inference_results_path}/results_comparison.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"\nComparison chart saved to {inference_results_path}/results_comparison.png")

## Save Results

In [ ]:
# Save detailed results to JSON
def get_per_class_metrics(y_true, y_pred, class_names):
    per_class = {}
    for cls in class_names:
        per_class[cls] = {
            "precision": float(precision_score(y_true, y_pred, average=None, labels=[cls])[0]),
            "recall": float(recall_score(y_true, y_pred, average=None, labels=[cls])[0]),
            "f1": float(f1_score(y_true, y_pred, average=None, labels=[cls])[0])
        }
    return per_class

results = {
    "experiment_number": EXP_NUMBER,
    "model_type": model_name,
    "dev_matched": {
        "accuracy": float(accuracy_matched),
        "precision_weighted": float(precision_matched),
        "recall_weighted": float(recall_matched),
        "f1_weighted": float(f1_matched),
        "per_class_metrics": get_per_class_metrics(y_matched, y_pred_matched, class_names),
        "num_samples": len(y_matched)
    },
    "dev_mismatched": {
        "accuracy": float(accuracy_mismatched),
        "precision_weighted": float(precision_mismatched),
        "recall_weighted": float(recall_mismatched),
        "f1_weighted": float(f1_mismatched),
        "per_class_metrics": get_per_class_metrics(y_mismatched, y_pred_mismatched, class_names),
        "num_samples": len(y_mismatched)
    }
}

# Save to JSON
results_file = f"{inference_results_path}/inference_results.json"
with open(results_file, 'w') as f:
    json.dump(results, f, indent=4)

print(f"\nResults saved to {results_file}")

# Also save comparison dataframe
results_comparison.to_csv(f"{inference_results_path}/results_comparison.csv", index=False)
print(f"Comparison table saved to {inference_results_path}/results_comparison.csv")

## Sample Predictions Analysis

In [ ]:
# Show some example predictions
def show_sample_predictions(df, y_true, y_pred, dataset_name, n_samples=10):
    print(f"\n{'='*80}")
    print(f"Sample Predictions - {dataset_name}")
    print(f"{'='*80}\n")
    
    sample_df = df.copy()
    sample_df['true_label'] = y_true.values
    sample_df['predicted_label'] = y_pred
    sample_df['correct'] = sample_df['true_label'] == sample_df['predicted_label']
    
    # Show some correct predictions
    print("\n--- Correct Predictions ---")
    correct_samples = sample_df[sample_df['correct']].sample(min(n_samples//2, len(sample_df[sample_df['correct']])))
    for idx, row in correct_samples.iterrows():
        print(f"\nSentence 1: {row['sentence1']}")
        print(f"Sentence 2: {row['sentence2']}")
        print(f"True Label: {row['true_label']} | Predicted: {row['predicted_label']} ✓")
        print("-" * 80)
    
    # Show some incorrect predictions
    print("\n--- Incorrect Predictions ---")
    incorrect_samples = sample_df[~sample_df['correct']].sample(min(n_samples//2, len(sample_df[~sample_df['correct']])))
    for idx, row in incorrect_samples.iterrows():
        print(f"\nSentence 1: {row['sentence1']}")
        print(f"Sentence 2: {row['sentence2']}")
        print(f"True Label: {row['true_label']} | Predicted: {row['predicted_label']} ✗")
        print("-" * 80)

show_sample_predictions(dev_matched, y_matched, y_pred_matched, "Dev Matched", n_samples=10)
show_sample_predictions(dev_mismatched, y_mismatched, y_pred_mismatched, "Dev Mismatched", n_samples=10)

## Summary

In [ ]:
print("\n" + "="*80)
print("INFERENCE SUMMARY")
print("="*80)
print(f"\nExperiment Number: {EXP_NUMBER}")
print(f"Model Type: {model_name}")
print(f"\nDev Matched:")
print(f"  - Samples: {len(y_matched)}")
print(f"  - Accuracy: {accuracy_matched:.4f}")
print(f"  - F1-Score: {f1_matched:.4f}")
print(f"\nDev Mismatched:")
print(f"  - Samples: {len(y_mismatched)}")
print(f"  - Accuracy: {accuracy_mismatched:.4f}")
print(f"  - F1-Score: {f1_mismatched:.4f}")
print(f"\nPerformance Drop (Matched → Mismatched):")
print(f"  - Accuracy: {(accuracy_matched - accuracy_mismatched)*100:.2f}%")
print(f"  - F1-Score: {(f1_matched - f1_mismatched)*100:.2f}%")
print(f"\nResults saved to: {inference_results_path}")
print("="*80)